<a href="https://colab.research.google.com/github/simar-rekhi/leskovec/blob/main/gcn_mutag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GCN Application on Toy Dataset / Mutag

In [10]:
# Install external libraries only
!pip install torch torchvision torchaudio
!pip install torch_geometric

# Standard Python imports (NO pip)
import os
import random
import argparse

from dataclasses import dataclass
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

We are using GCN to gather graphical classfication of toy dataset i.e., MUTAG via PyTorch Geometric.

- config
- data loading
- model config
- train/eval
- main / init

In [11]:
# seed setting (defacto 42)
def set_seed(seed : int = 42) -> None:
  random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)


# training configuration
@dataclass
class TrainConfig:
  dataset: str = "MUTAG"
  root: str = "./data/TUDataset"
  batch_size: int = 32
  lr: float = 5e-4
  hidden_dim: int = 64
  num_layers: int = 3
  dropout: float = 0.5
  epochs: int = 100
  seed: int = 42
  device: str = "cuda" if torch.cuda.is_available() else "cpu"
  train_ratio: float = 0.8  # simple random split for toy datasets

In [12]:
# data loading and splitting

Loading part of the Dataset:
- load graphical classification
- MUTAG has node_features and labels


In [13]:
def load_dataset(cfg: TrainConfig) -> TUDataset:

  dataset = TUDataset(root=cfg.root, name=cfg.dataset)
  return dataset


def split_dataset(dataset: TUDataset, train_ratio: float, seed: int) -> Tuple[list, list]:
  indices = list(range(len(dataset)))
  rng = random.Random(seed)
  rng.shuffle(indices)

  split = int(train_ratio * len(indices))
  train_idx = indices[:split]
  test_idx = indices[split:]

  train_set = [dataset[i] for i in train_idx]
  test_set = [dataset[i] for i in test_idx]
  return train_set, test_set



# data loader would create a PyG dataloader, batches the graphs together to a concatenated batch vector
# map each node to graph
def make_loaders(train_set: list, test_set: list, batch_size: int) -> Tuple[DataLoader, DataLoader]:
  train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
  test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
  return train_loader, test_loader


Model Definition

In [14]:
class GCNGraphClassifier(nn.Module):

  def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, num_layers: int, dropout: float):
        super().__init__()
        assert num_layers >= 2, "Use at least 2 layers (1+ for features, 1+ for depth)."

        self.dropout = dropout

        # First GCN layer always maps out the features
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(in_dim, hidden_dim))

        # Middle GCN layers keep hidden_dim
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))

        # Last GCN layer (still hidden_dim; pooling happens after)
        self.convs.append(GCNConv(hidden_dim, hidden_dim))

        # Graph-level classifier head (after pooling)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
        )


  def forward(self, x, edge_index, batch):

        # Node-level message passing
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Pool node embeddings into one graph embedding per graph in batch
        g = global_mean_pool(x, batch)  # [num_graphs_in_batch, hidden_dim]

        # Graph-level classification
        logits = self.classifier(g)     # [num_graphs_in_batch, out_dim]
        return logits

Train / Eval Loops

In [15]:
def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, device: str) -> float:

    model.train()
    total_loss = 0.0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)

        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index, batch.batch)   # graph-level logits
        loss = F.cross_entropy(logits, batch.y)                  # batch.y has graph labels
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch.num_graphs
        total_graphs += batch.num_graphs

    return total_loss / max(total_graphs, 1)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: str) -> float:

    model.eval()
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index, batch.batch)
        preds = logits.argmax(dim=-1)
        correct += int((preds == batch.y).sum().item())
        total += batch.num_graphs

    return correct / max(total, 1)

In [16]:
def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--dataset", type=str, default="MUTAG")
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--epochs", type=int, default=100)

    args = parser.parse_args()

    return args

In [17]:
def main():
    cfg = parse_args()
    set_seed(cfg.seed)

    print(f"Device: {cfg.device}")
    print(f"Loading dataset: {cfg.dataset}")

    dataset = load_dataset(cfg)
    print(f"Num graphs: {len(dataset)} | Num classes: {dataset.num_classes} | Num node features: {dataset.num_features}")

    train_set, test_set = split_dataset(dataset, cfg.train_ratio, cfg.seed)
    train_loader, test_loader = make_loaders(train_set, test_set, cfg.batch_size)

    model = build_model(dataset, cfg).to(cfg.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_test_acc = 0.0

    for epoch in range(1, cfg.epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, cfg.device)
        train_acc = evaluate(model, train_loader, cfg.device)
        test_acc = evaluate(model, test_loader, cfg.device)

        best_test_acc = max(best_test_acc, test_acc)

        if epoch == 1 or epoch % 10 == 0 or epoch == cfg.epochs:
            print(
                f"Epoch {epoch:03d}/{cfg.epochs} | "
                f"Loss: {loss:.4f} | Train Acc: {train_acc:.3f} | Test Acc: {test_acc:.3f} | Best: {best_test_acc:.3f}"
            )

    print(f"Done. Best Test Accuracy: {best_test_acc:.3f}")


if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h] [--dataset DATASET]
                                [--batch_size BATCH_SIZE] [--epochs EPOCHS]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-75802719-ff94-4630-b2e4-beecb95822cc.json


SystemExit: 2